In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Make plots look professional
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Create reports folder for saving plots
import os
os.makedirs('../reports', exist_ok=True)

print("✅ All libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Load dataset
df = pd.read_csv('../data/creditcard.csv')

print(f"Dataset shape: {df.shape}")
print(f"Total transactions: {df.shape[0]:,}")
print(f"Total features: {df.shape[1]}")
print(f"\nMemory used: {df.memory_usage().sum() / 1024**2:.2f} MB")

# Show first 5 rows
df.head()

In [ ]:
# Data types and missing values check
print("=" * 60)
print("DATA TYPES & MISSING VALUES")
print("=" * 60)
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})
print(info_df)

print("\n" + "=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
df.describe().round(2)

In [ ]:
# Calculate class distribution
class_counts = df['Class'].value_counts()
class_percent = df['Class'].value_counts(normalize=True) * 100

print("🎯 CLASS DISTRIBUTION")
print("=" * 60)
print(f"Legitimate transactions: {class_counts[0]:,} ({class_percent[0]:.3f}%)")
print(f"Fraudulent transactions: {class_counts[1]:,} ({class_percent[1]:.3f}%)")
print(f"Imbalance ratio: 1 fraud per {class_counts[0] // class_counts[1]} legit transactions")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
class_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Transaction Count by Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class (0 = Legitimate, 1 = Fraud)')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_yscale('log')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
for i, v in enumerate(class_counts):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=colors, startangle=90, explode=(0, 0.1), shadow=True)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ INSIGHT: With 0.172% fraud, a dumb model saying 'never fraud' ")
print("   would score 99.83% accuracy but catch 0 frauds. This is why we")
print("   use Precision, Recall, F1, and ROC-AUC — not just accuracy.")

In [ ]:
# Split into fraud vs legit
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print("💰 TRANSACTION AMOUNT ANALYSIS")
print("=" * 60)
print("\nFraudulent transactions:")
print(fraud['Amount'].describe().round(2))
print("\nLegitimate transactions:")
print(legit['Amount'].describe().round(2))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram — fraud
axes[0, 0].hist(fraud['Amount'], bins=50, color='#e74c3c', edgecolor='black')
axes[0, 0].set_title('Fraud — Amount Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Frequency')

# Histogram — legit
axes[0, 1].hist(legit['Amount'], bins=50, color='#2ecc71', edgecolor='black')
axes[0, 1].set_title('Legitimate — Amount Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Amount ($)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_yscale('log')

# Boxplot comparison (limited to $500 for readability)
sns.boxplot(data=df[df['Amount'] < 500], x='Class', y='Amount', ax=axes[1, 0], palette=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Amount Comparison (< $500)', fontweight='bold')
axes[1, 0].set_xticklabels(['Legitimate', 'Fraud'])

# Top amounts
axes[1, 1].bar(['Fraud Max', 'Fraud Mean', 'Legit Max', 'Legit Mean'],
               [fraud['Amount'].max(), fraud['Amount'].mean(),
                legit['Amount'].max(), legit['Amount'].mean()],
               color=['#e74c3c', '#c0392b', '#2ecc71', '#27ae60'])
axes[1, 1].set_title('Amount Statistics', fontweight='bold')
axes[1, 1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.savefig('../reports/02_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🔍 INSIGHT: Fraud avg = ${fraud['Amount'].mean():.2f} vs Legit avg = ${legit['Amount'].mean():.2f}")
print(f"   Fraudsters tend to use SMALLER amounts to avoid detection.")

In [ ]:
# Convert Time (seconds from first transaction) to hours
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transactions by hour
fraud_by_hour = df[df['Class'] == 1]['Hour'].value_counts().sort_index()
legit_by_hour = df[df['Class'] == 0]['Hour'].value_counts().sort_index()

axes[0].plot(legit_by_hour.index, legit_by_hour.values, marker='o', color='#2ecc71', label='Legitimate', linewidth=2)
axes[0].set_title('Legitimate Transactions by Hour', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].plot(fraud_by_hour.index, fraud_by_hour.values, marker='o', color='#e74c3c', label='Fraud', linewidth=2)
axes[1].set_title('Fraudulent Transactions by Hour', fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/03_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("🔍 INSIGHT: Legit transactions follow day/night patterns.")
print("   Fraud is more evenly distributed — fraudsters don't sleep!")

In [ ]:
plt.figure(figsize=(20, 14))
corr = df.drop('Hour', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 7})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top features correlated with fraud
print("\n🎯 TOP 10 FEATURES MOST CORRELATED WITH FRAUD")
print("=" * 60)
fraud_corr = corr['Class'].drop('Class').abs().sort_values(ascending=False)
print(fraud_corr.head(10).round(4))

In [ ]:
# The top features from correlation analysis
top_features = ['V17', 'V14', 'V12', 'V10', 'V16', 'V11']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].hist(legit[feat], bins=50, alpha=0.6, label='Legit', color='#2ecc71', density=True)
    axes[i].hist(fraud[feat], bins=50, alpha=0.6, label='Fraud', color='#e74c3c', density=True)
    axes[i].set_title(f'Feature {feat}', fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.suptitle('Distribution of Top Fraud-Discriminative Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/05_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

print("🔍 INSIGHT: Features V17, V14, V12, V10 show CLEAR separation between")
print("   fraud (red) and legitimate (green) distributions. These will be")
print("   the most important signals for our ML models.")

In [ ]:
print("=" * 70)
print("📋 EDA SUMMARY — KEY FINDINGS FOR MODELING")
print("=" * 70)
print(f"""
1. DATASET SIZE
   • Total transactions: {len(df):,}
   • Features: {df.shape[1] - 2}
   • Missing values: {df.isnull().sum().sum()}

2. CLASS IMBALANCE (CRITICAL)
   • Legitimate: {class_counts[0]:,} ({class_percent[0]:.3f}%)
   • Fraud:      {class_counts[1]:,} ({class_percent[1]:.3f}%)
   • Ratio:      1 : {class_counts[0] // class_counts[1]}
   ⚠️  MUST use SMOTE or class_weight to handle this

3. AMOUNT PATTERNS
   • Fraud avg: ${fraud['Amount'].mean():.2f}   |   Legit avg: ${legit['Amount'].mean():.2f}
   • Fraudsters prefer smaller amounts

4. TEMPORAL PATTERNS
   • Legit follows day/night cycle
   • Fraud is evenly distributed across hours

5. MOST IMPORTANT FEATURES
   • V17, V14, V12, V10 have highest correlation with fraud
   • Clear visual separation in their distributions

6. METRICS TO USE (NOT accuracy alone!)
   • Precision  — of predicted frauds, how many were real?
   • Recall     — of real frauds, how many did we catch?
   • F1-Score   — balance of precision and recall
   • ROC-AUC    — overall ranking quality



✅ READY FOR PHASE 4: Preprocessing + SMOTE
""")